# Create Genie Space & Write State

Creates a Databricks Genie Space from the generated tables and writes `state.json` to a volume for the app to read at startup.

In [ ]:
dbutils.widgets.text("catalog", "startups_catalog")
dbutils.widgets.text("schema", "dw_genie")
dbutils.widgets.text("display_name", "Acme Analytics")
dbutils.widgets.text("company_name", "Acme Corporation")
dbutils.widgets.text("company_description", "Global leader in industrial supplies and equipment, serving customers across 12 industries worldwide.")
dbutils.widgets.text("primary_color", "#1a73e8")
dbutils.widgets.text("secondary_color", "#ea4335")
dbutils.widgets.text("existing_space_id", "")
dbutils.widgets.text("warehouse_id", "")

CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
DISPLAY_NAME = dbutils.widgets.get("display_name")
COMPANY_NAME = dbutils.widgets.get("company_name")
COMPANY_DESC = dbutils.widgets.get("company_description")
PRIMARY_COLOR = dbutils.widgets.get("primary_color")
SECONDARY_COLOR = dbutils.widgets.get("secondary_color")
EXISTING_SPACE_ID = dbutils.widgets.get("existing_space_id").strip()
WAREHOUSE_ID = dbutils.widgets.get("warehouse_id").strip()

VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/raw_data"
STATE_PATH = f"{VOLUME_PATH}/state.json"

TABLE_NAMES = ["customers", "products", "orders", "order_items"]
TABLE_IDENTIFIERS = [f"{CATALOG}.{SCHEMA}.{t}" for t in TABLE_NAMES]

SAMPLE_QUESTIONS = [
    "What were total sales last month?",
    "Who are our top 10 customers by revenue?",
    "What is the average order value by customer tier?",
    "Which product categories have the highest margins?",
    "Show me customer locations with their latitude and longitude",
]

print(f"Tables: {TABLE_IDENTIFIERS}")
print(f"Existing space ID: '{EXISTING_SPACE_ID}' (empty = create new)")

In [ ]:
import json
import requests

# Get workspace host and token for REST API calls
ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
host = ctx.apiUrl().get()
token = ctx.apiToken().get()
headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

print(f"Workspace: {host}")

In [ ]:
if EXISTING_SPACE_ID:
    # Use existing Genie Space
    print(f"Using existing Genie Space: {EXISTING_SPACE_ID}")
    resp = requests.get(f"{host}/api/2.0/genie/spaces/{EXISTING_SPACE_ID}", headers=headers)
    resp.raise_for_status()
    space = resp.json()
    space_id = EXISTING_SPACE_ID
    display_name = space.get("display_name", DISPLAY_NAME)
    table_ids = space.get("table_identifiers", TABLE_IDENTIFIERS)
    print(f"Space: {display_name}, Tables: {table_ids}")
else:
    # Create new Genie Space via REST API
    print(f"Creating Genie Space: {DISPLAY_NAME}")
    payload = {
        "display_name": DISPLAY_NAME,
        "description": COMPANY_DESC,
        "table_identifiers": TABLE_IDENTIFIERS,
        "sample_questions": SAMPLE_QUESTIONS,
    }
    if WAREHOUSE_ID:
        payload["warehouse_id"] = WAREHOUSE_ID
    resp = requests.post(f"{host}/api/2.0/genie/spaces", headers=headers, json=payload)
    resp.raise_for_status()
    space = resp.json()
    space_id = space["space_id"]
    display_name = DISPLAY_NAME
    table_ids = TABLE_IDENTIFIERS
    print(f"Created Genie Space: {space_id}")

In [ ]:
# Get table comments for state via REST API
tables_info = []
for tid in table_ids:
    table_name = tid.split(".")[-1]
    try:
        resp = requests.get(f"{host}/api/2.1/unity-catalog/tables/{tid}", headers=headers)
        resp.raise_for_status()
        info = resp.json()
        comment = info.get("comment", "")
    except Exception:
        comment = ""
    tables_info.append({"full_name": tid, "table_name": table_name, "comment": comment})

print(f"Table info: {[t['table_name'] for t in tables_info]}")

In [ ]:
state = {
    "space_id": space_id,
    "display_name": display_name,
    "catalog": CATALOG,
    "schema_name": SCHEMA,
    "warehouse_id": WAREHOUSE_ID,
    "tables": tables_info,
    "sample_questions": SAMPLE_QUESTIONS,
    "branding": {
        "company_name": COMPANY_NAME,
        "description": COMPANY_DESC,
        "logo_path": "",
        "primary_color": PRIMARY_COLOR,
        "secondary_color": SECONDARY_COLOR,
    },
}

state_json = json.dumps(state, indent=2)
dbutils.fs.put(STATE_PATH, state_json, overwrite=True)

print(f"State written to {STATE_PATH}")
print(state_json)